# Fraud Detection & Transaction Risk Modeling

## Model Development

This notebook develops and compares predictive models for transaction fraud
detection.

The modelling objective is to estimate the probability that an individual
transaction is fraudulent at the time the transaction occurs.

The model development process uses a chronological train-validation split to
simulate real-world deployment, where historical transactions are used to
predict future transactions.

Models will be evaluated using fraud-focused metrics including precision,
recall, F1-score, PR-AUC, and ROC-AUC.

The modelling pipeline will avoid using future validation information during
training or preprocessing.

## Step 25B — Modelling Configuration

The modelling notebook uses the finalized feature definitions and
preprocessing strategy developed in the feature-engineering stage.

The training data remains chronological, with earlier transactions used for
training and later transactions reserved for validation.

Chunked processing is used where necessary to control memory usage on the
local development environment.

In [1]:
# Step 25A — Load finalized modelling datasets

import pandas as pd
import numpy as np
import gc

train_path = "../data/processed/train_features.csv"
valid_path = "../data/processed/validation_features.csv"

print("=" * 60)
print("LOADING FINAL MODELLING DATA")
print("=" * 60)

train_df = pd.read_csv(train_path)
valid_df = pd.read_csv(valid_path)

print("\nTraining shape:", train_df.shape)
print("Validation shape:", valid_df.shape)

print("\nTarget distribution — Training:")
print(train_df["isFraud"].value_counts())
print("\nTraining fraud rate:", train_df["isFraud"].mean())

print("\nTarget distribution — Validation:")
print(valid_df["isFraud"].value_counts())
print("\nValidation fraud rate:", valid_df["isFraud"].mean())

print("\nColumn consistency:")
print("Same columns:", train_df.columns.equals(valid_df.columns))

print("\n" + "=" * 60)
print("DATA LOADING COMPLETED")
print("=" * 60)

LOADING FINAL MODELLING DATA

Training shape: (472432, 434)
Validation shape: (118108, 434)

Target distribution — Training:
isFraud
0    455833
1     16599
Name: count, dtype: int64

Training fraud rate: 0.03513521522674162

Target distribution — Validation:
isFraud
0    114044
1      4064
Name: count, dtype: int64

Validation fraud rate: 0.034409184813899145

Column consistency:
Same columns: True

DATA LOADING COMPLETED


In [2]:
# Step 25B — Separate features and target

target = "isFraud"

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

print("=" * 60)
print("FEATURE / TARGET SEPARATION")
print("=" * 60)

print("\nX_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nX_valid shape:", X_valid.shape)
print("y_valid shape:", y_valid.shape)

print("\nTarget in X_train:", target in X_train.columns)
print("Target in X_valid:", target in X_valid.columns)

print("\nFeature columns identical:",
      X_train.columns.equals(X_valid.columns))

print("\nNumber of numerical features:",
      X_train.select_dtypes(include=["int64", "float64"]).shape[1])

print("Number of categorical features:",
      X_train.select_dtypes(include=["object"]).shape[1])

print("\n" + "=" * 60)
print("FEATURE / TARGET SEPARATION COMPLETED")
print("=" * 60)

FEATURE / TARGET SEPARATION

X_train shape: (472432, 433)
y_train shape: (472432,)

X_valid shape: (118108, 433)
y_valid shape: (118108,)

Target in X_train: False
Target in X_valid: False

Feature columns identical: True

Number of numerical features: 404
Number of categorical features: 29

FEATURE / TARGET SEPARATION COMPLETED


In [3]:
# Step 25C — Reconstruct and verify final model feature groups

# Features intentionally removed because they were near-constant
near_constant_features = [
    "V1",
    "V14",
    "V41",
    "V65",
    "V88",
    "V107",
    "V305"
]

# Feature removed because it is perfectly redundant with D4
redundant_features = [
    "D12"
]

# Final numerical features
final_model_numerical_features = [
    col for col in X_train.select_dtypes(include=["int64", "float64"]).columns
    if col not in near_constant_features + redundant_features
]

# Frequency-encoded categorical features
frequency_features = [
    "DeviceInfo_freq",
    "id_33_freq"
]

# Grouped categorical features
grouped_features = [
    "id_31_grouped",
    "id_30_grouped"
]

# Remaining categorical features for one-hot encoding
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

final_model_categorical_features = [
    col for col in categorical_features
    if col not in ["DeviceInfo", "id_33", "id_31", "id_30"]
]

# Add the engineered categorical representations
final_model_categorical_features = (
    final_model_categorical_features + grouped_features
)

# Frequency features are numerical inputs
final_model_numerical_features = (
    final_model_numerical_features + frequency_features
)

# Remove duplicate names if any
final_model_numerical_features = list(dict.fromkeys(final_model_numerical_features))
final_model_categorical_features = list(dict.fromkeys(final_model_categorical_features))

final_model_features = (
    final_model_numerical_features +
    final_model_categorical_features
)

print("=" * 60)
print("FINAL MODEL FEATURE AUDIT")
print("=" * 60)

print("\nFinal numerical inputs:", len(final_model_numerical_features))
print("Final categorical inputs:", len(final_model_categorical_features))
print("Total final model inputs:", len(final_model_features))

print("\nExcluded near-constant features:")
print(near_constant_features)

print("\nExcluded redundant features:")
print(redundant_features)

print("\nFrequency-encoded features:")
print(frequency_features)

print("\nGrouped categorical features:")
print(grouped_features)

print("\nTarget included:", target in final_model_features)

print("Duplicate final feature names:",
      len(final_model_features) - len(set(final_model_features)))

print(
    "\nAll final features available:",
    all(col in X_train.columns for col in final_model_features)
)

print("\n" + "=" * 60)
print("FINAL MODEL FEATURE AUDIT COMPLETED")
print("=" * 60)

FINAL MODEL FEATURE AUDIT

Final numerical inputs: 404
Final categorical inputs: 29
Total final model inputs: 433

Excluded near-constant features:
['V1', 'V14', 'V41', 'V65', 'V88', 'V107', 'V305']

Excluded redundant features:
['D12']

Frequency-encoded features:
['DeviceInfo_freq', 'id_33_freq']

Grouped categorical features:
['id_31_grouped', 'id_30_grouped']

Target included: False
Duplicate final feature names: 0

All final features available: True

FINAL MODEL FEATURE AUDIT COMPLETED


In [4]:
# Step 25D — Identify missing and extra model features

expected_features = set(final_model_features)
actual_features = set(X_train.columns)

missing_features = sorted(expected_features - actual_features)
extra_features = sorted(actual_features - expected_features)

print("=" * 60)
print("FINAL MODEL FEATURE AVAILABILITY CHECK")
print("=" * 60)

print("\nExpected final model features:", len(expected_features))
print("Actual CSV features:", len(actual_features))

print("\nMissing expected features:", len(missing_features))

if missing_features:
    print("\nMissing features:")
    for feature in missing_features:
        print("-", feature)

print("\nExtra CSV features:", len(extra_features))

if extra_features:
    print("\nExtra features:")
    for feature in extra_features:
        print("-", feature)

print("\n" + "=" * 60)
print("FEATURE AVAILABILITY CHECK COMPLETED")
print("=" * 60)

FINAL MODEL FEATURE AVAILABILITY CHECK

Expected final model features: 433
Actual CSV features: 433

Missing expected features: 0

Extra CSV features: 0

FEATURE AVAILABILITY CHECK COMPLETED


In [5]:
# Step 25E — Reload the corrected final modelling datasets

import pandas as pd
import numpy as np
import gc

train_path = "../data/processed/train_features.csv"
valid_path = "../data/processed/validation_features.csv"

print("=" * 60)
print("LOADING CORRECTED MODELLING DATA")
print("=" * 60)

# Clear old references first
for variable in ["train_df", "valid_df", "X_train", "X_valid", "y_train", "y_valid"]:
    if variable in globals():
        del globals()[variable]

gc.collect()

# Load corrected datasets
train_df = pd.read_csv(train_path)
valid_df = pd.read_csv(valid_path)

# Separate target
target = "isFraud"

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

print("\nTraining shape:", train_df.shape)
print("Validation shape:", valid_df.shape)

print("\nX_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)

print("\ny_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)

print("\nTarget in X_train:", target in X_train.columns)
print("Target in X_valid:", target in X_valid.columns)

print("\nFeature columns identical:",
      X_train.columns.equals(X_valid.columns))

print("\nTraining fraud rate:", y_train.mean())
print("Validation fraud rate:", y_valid.mean())

print("\n" + "=" * 60)
print("CORRECTED MODELLING DATA LOADED")
print("=" * 60)

LOADING CORRECTED MODELLING DATA

Training shape: (472432, 434)
Validation shape: (118108, 434)

X_train shape: (472432, 433)
X_valid shape: (118108, 433)

y_train shape: (472432,)
y_valid shape: (118108,)

Target in X_train: False
Target in X_valid: False

Feature columns identical: True

Training fraud rate: 0.03513521522674162
Validation fraud rate: 0.034409184813899145

CORRECTED MODELLING DATA LOADED


## Step 26 — Final Preprocessing Pipeline

The modelling dataset contains 433 finalized input features.

Numerical features will use training-fitted median imputation followed by
standardization. Categorical features will retain missingness as an explicit
category and use one-hot encoding with safe handling of unseen validation
categories.

The preprocessing pipeline will be fitted only on the training period.
Validation data will only be transformed after the preprocessing parameters
have been learned.

Sparse output is used for categorical expansion to control memory usage.

In [6]:
# Step 26A — Define and audit final preprocessing inputs

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np

# ------------------------------------------------------------
# Final numerical features
# ------------------------------------------------------------

near_constant_features = [
    "V1",
    "V14",
    "V41",
    "V65",
    "V88",
    "V107",
    "V305"
]

redundant_features = [
    "D12"
]

# Frequency-encoded features are numerical
frequency_features = [
    "DeviceInfo_freq",
    "id_33_freq"
]

# Identify numerical columns currently present in X_train
base_numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Remove features intentionally excluded
base_numerical_features = [
    col for col in base_numerical_features
    if col not in near_constant_features
    and col not in redundant_features
    and col not in frequency_features
]

final_model_numerical_features = (
    base_numerical_features + frequency_features
)

# ------------------------------------------------------------
# Final categorical features
# ------------------------------------------------------------

categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Raw high-cardinality columns were replaced by engineered versions
categorical_to_replace = [
    "DeviceInfo",
    "id_33",
    "id_31",
    "id_30"
]

remaining_categorical_features = [
    col for col in categorical_columns
    if col not in categorical_to_replace
]

grouped_features = [
    "id_31_grouped",
    "id_30_grouped"
]

final_model_categorical_features = (
    remaining_categorical_features + grouped_features
)

# Remove duplicates while preserving order
final_model_numerical_features = list(
    dict.fromkeys(final_model_numerical_features)
)

final_model_categorical_features = list(
    dict.fromkeys(final_model_categorical_features)
)

final_model_features = (
    final_model_numerical_features +
    final_model_categorical_features
)

# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

missing_features = [
    col for col in final_model_features
    if col not in X_train.columns
]

unexpected_features = [
    col for col in X_train.columns
    if col not in final_model_features
]

print("=" * 60)
print("FINAL PREPROCESSING INPUT AUDIT")
print("=" * 60)

print("\nNumerical inputs:", len(final_model_numerical_features))
print("Categorical inputs:", len(final_model_categorical_features))
print("Total preprocessing inputs:", len(final_model_features))

print("\nMissing expected features:", len(missing_features))

if missing_features:
    print(missing_features)

print("\nExcluded from preprocessing:", len(unexpected_features))

if unexpected_features:
    print(unexpected_features)

print("\nAll final features available:",
      len(missing_features) == 0)

print("\nFeature count matches expected 433:",
      len(final_model_features) == 433)

print("\n" + "=" * 60)
print("PREPROCESSING INPUT AUDIT COMPLETED")
print("=" * 60)

FINAL PREPROCESSING INPUT AUDIT

Numerical inputs: 404
Categorical inputs: 29
Total preprocessing inputs: 433

Missing expected features: 0

Excluded from preprocessing: 0

All final features available: True

Feature count matches expected 433: True

PREPROCESSING INPUT AUDIT COMPLETED


In [7]:
# Step 26B — Construct preprocessing pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np
import scipy.sparse as sparse

# ------------------------------------------------------------
# Numerical preprocessing
# ------------------------------------------------------------

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# ------------------------------------------------------------
# Categorical preprocessing
# ------------------------------------------------------------

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="__MISSING__"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32
        )
    )
])

# ------------------------------------------------------------
# Combined preprocessing
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            final_model_numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            final_model_categorical_features
        )
    ],
    remainder="drop",
    sparse_threshold=1.0
)

print("=" * 60)
print("PREPROCESSING PIPELINE CREATED")
print("=" * 60)

print("\nNumerical features:", len(final_model_numerical_features))
print("Categorical features:", len(final_model_categorical_features))
print("Total input features:", len(final_model_features))

print("\nNumerical transformation:")
print("  Median imputation → StandardScaler")

print("\nCategorical transformation:")
print("  Missing category → OneHotEncoder")

print("\nUnknown categories:")
print("  handle_unknown='ignore'")

print("\nOutput:")
print("  Sparse matrix")

print("\n" + "=" * 60)
print("PIPELINE READY")
print("=" * 60)

PREPROCESSING PIPELINE CREATED

Numerical features: 404
Categorical features: 29
Total input features: 433

Numerical transformation:
  Median imputation → StandardScaler

Categorical transformation:
  Missing category → OneHotEncoder

Unknown categories:
  handle_unknown='ignore'

Output:
  Sparse matrix

PIPELINE READY


In [ ]:
# Step 26C — Test preprocessing on a small training sample

sample_size = 10_000

X_train_sample = X_train.iloc[:sample_size]

print("=" * 60)
print("SAMPLE PREPROCESSING TEST")
print("=" * 60)

print("\nInput sample shape:", X_train_sample.shape)

# Fit and transform ONLY the sample for this technical test
X_sample_transformed = preprocessor.fit_transform(X_train_sample)

print("\nTransformed shape:", X_sample_transformed.shape)
print("Output type:", type(X_sample_transformed).__name__)

if sparse.issparse(X_sample_transformed):
    total_values = (
        X_sample_transformed.shape[0] *
        X_sample_transformed.shape[1]
    )

    print("Output format:", X_sample_transformed.getformat())
    print("Output dtype:", X_sample_transformed.dtype)
    print("Non-zero values:", X_sample_transformed.nnz)
    print("Total matrix values:", total_values)

    sparsity = 1 - (
        X_sample_transformed.nnz / total_values
    )

    print("Sparsity:", round(sparsity, 4))

else:
    print("WARNING: Output is dense.")

print("\n" + "=" * 60)
print("SAMPLE PREPROCESSING TEST COMPLETED")
print("=" * 60)


SAMPLE PREPROCESSING TEST

Input sample shape: (10000, 433)

Transformed shape: (10000, 682)
Output type: csr_matrix
Output format: csr
Output dtype: float64
Non-zero values: 4300058
Total matrix values: 6820000
Sparsity: 0.3695

SAMPLE PREPROCESSING TEST COMPLETED


## Step 27 — Final Preprocessing Fit

The final preprocessing pipeline is fitted using the complete chronological
training dataset.

No information from the validation period is used to estimate numerical
imputation values, scaling parameters, or categorical encoding categories.

After fitting, the training data is transformed into a sparse matrix suitable
for memory-efficient modelling.

The validation data will only be transformed using the already-fitted
training preprocessing pipeline.

In [10]:
# Step 27A — Fit final preprocessing pipeline on full training data

import time
import gc
import numpy as np
import scipy.sparse as sparse

# ------------------------------------------------------------
# Create a FRESH final preprocessor
# ------------------------------------------------------------

final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            final_model_numerical_features
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="__MISSING__"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=True,
                        dtype=np.float32
                    )
                )
            ]),
            final_model_categorical_features
        )
    ],
    remainder="drop",
    sparse_threshold=1.0
)

print("=" * 60)
print("FITTING FINAL PREPROCESSOR")
print("=" * 60)

print("\nTraining rows:", len(X_train))
print("Training features:", X_train.shape[1])

# ------------------------------------------------------------
# Fit only on training data
# ------------------------------------------------------------

start_time = time.time()

final_preprocessor.fit(X_train)

elapsed_time = time.time() - start_time

print("\nFinal preprocessor fitted successfully.")
print("Time taken: {:.2f} minutes".format(elapsed_time / 60))

print("\n" + "=" * 60)
print("FINAL PREPROCESSOR FIT COMPLETED")
print("=" * 60)

FITTING FINAL PREPROCESSOR

Training rows: 472432
Training features: 433

Final preprocessor fitted successfully.
Time taken: 2.36 minutes

FINAL PREPROCESSOR FIT COMPLETED


## Step 27B — Transform Full Training Data

The fitted preprocessing pipeline is now used to transform the complete
training dataset.

The transformation produces a sparse matrix so that one-hot encoded
categorical variables do not require a large dense matrix.

The resulting matrix will be used as the input for the baseline model.

In [11]:
# Step 27B — Transform full training data

import time
import gc
import scipy.sparse as sparse

print("=" * 60)
print("TRANSFORMING FULL TRAINING DATA")
print("=" * 60)

print("\nInput shape:", X_train.shape)

start_time = time.time()

X_train_transformed = final_preprocessor.transform(X_train)

elapsed_time = time.time() - start_time

print("\nTransformation completed.")
print("Time taken: {:.2f} minutes".format(elapsed_time / 60))

print("\nTransformed shape:", X_train_transformed.shape)
print("Output type:", type(X_train_transformed).__name__)

if sparse.issparse(X_train_transformed):

    print("Output format:", X_train_transformed.getformat())
    print("Output dtype:", X_train_transformed.dtype)
    print("Non-zero values:", X_train_transformed.nnz)

    total_values = (
        X_train_transformed.shape[0] *
        X_train_transformed.shape[1]
    )

    sparsity = 1 - (
        X_train_transformed.nnz / total_values
    )

    print("Total matrix values:", total_values)
    print("Sparsity:", round(sparsity, 4))

    # Approximate sparse matrix memory
    memory_mb = (
        X_train_transformed.data.nbytes
        + X_train_transformed.indices.nbytes
        + X_train_transformed.indptr.nbytes
    ) / (1024 ** 2)

    print("Approximate sparse matrix memory:",
          round(memory_mb, 2), "MB")

else:
    print("WARNING: Output is dense.")

print("\n" + "=" * 60)
print("FULL TRAINING TRANSFORMATION COMPLETED")
print("=" * 60)

TRANSFORMING FULL TRAINING DATA

Input shape: (472432, 433)

Transformation completed.
Time taken: 1.43 minutes

Transformed shape: (472432, 768)
Output type: csr_matrix
Output format: csr
Output dtype: float64
Non-zero values: 204563056
Total matrix values: 362827776
Sparsity: 0.4362
Approximate sparse matrix memory: 2342.84 MB

FULL TRAINING TRANSFORMATION COMPLETED


In [12]:
# Step 27C — Validate transformed training matrix

import numpy as np
import scipy.sparse as sparse

print("=" * 60)
print("VALIDATING TRANSFORMED TRAINING MATRIX")
print("=" * 60)

print("\nMatrix shape:", X_train_transformed.shape)
print("Target shape:", y_train.shape)
print("Sparse matrix:", sparse.issparse(X_train_transformed))

# Row alignment
print("\nRow alignment:",
      X_train_transformed.shape[0] == len(y_train))

# Expected transformed feature count
print("Transformed feature count:",
      X_train_transformed.shape[1])

# Check NaN / infinite values only on stored sparse values
data = X_train_transformed.data

nan_count = np.isnan(data).sum()
inf_count = np.isinf(data).sum()

print("\nNaN values:", nan_count)
print("Infinite values:", inf_count)

# Basic value range
print("\nMinimum stored value:", data.min())
print("Maximum stored value:", data.max())

print("\n" + "=" * 60)
print("TRANSFORMED TRAINING MATRIX VALIDATION COMPLETED")
print("=" * 60)

VALIDATING TRANSFORMED TRAINING MATRIX

Matrix shape: (472432, 768)
Target shape: (472432,)
Sparse matrix: True

Row alignment: True
Transformed feature count: 768

NaN values: 0
Infinite values: 0

Minimum stored value: -116.57067461778234
Maximum stored value: 546.4610657019065

TRANSFORMED TRAINING MATRIX VALIDATION COMPLETED


# Step 28A — Train the Logistic Regression baseline

In [13]:
# Step 28A — Logistic Regression baseline

import time
import gc

from sklearn.linear_model import LogisticRegression

print("=" * 60)
print("TRAINING LOGISTIC REGRESSION BASELINE")
print("=" * 60)

print("\nTraining matrix shape:", X_train_transformed.shape)
print("Training fraud rate:", round(y_train.mean(), 6))

logistic_model = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    C=1.0,
    max_iter=100,
    random_state=42
)

print("\nModel configuration:")
print("Solver:", logistic_model.solver)
print("Class weight:", logistic_model.class_weight)
print("C:", logistic_model.C)
print("Maximum iterations:", logistic_model.max_iter)

print("\nStarting training...")

start_time = time.time()

logistic_model.fit(
    X_train_transformed,
    y_train
)

elapsed_time = time.time() - start_time

print("\nTraining completed.")
print("Time taken: {:.2f} minutes".format(elapsed_time / 60))

print("\nIterations used:",
      logistic_model.n_iter_[0])

print("\n" + "=" * 60)
print("LOGISTIC REGRESSION TRAINING COMPLETED")
print("=" * 60)

TRAINING LOGISTIC REGRESSION BASELINE

Training matrix shape: (472432, 768)
Training fraud rate: 0.035135

Model configuration:
Solver: saga
Class weight: balanced
C: 1.0
Maximum iterations: 100

Starting training...

Training completed.
Time taken: 2.33 minutes

Iterations used: 100

LOGISTIC REGRESSION TRAINING COMPLETED


/Users/raghavendradivate/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


# Step 28B — Transform validation data

In [14]:
# Step 28B — Transform full validation data

import time
import scipy.sparse as sparse

print("=" * 60)
print("TRANSFORMING FULL VALIDATION DATA")
print("=" * 60)

print("\nInput shape:", X_valid.shape)

start_time = time.time()

X_valid_transformed = final_preprocessor.transform(X_valid)

elapsed_time = time.time() - start_time

print("\nTransformation completed.")
print("Time taken: {:.2f} minutes".format(elapsed_time / 60))

print("\nTransformed shape:", X_valid_transformed.shape)
print("Output type:", type(X_valid_transformed).__name__)

if sparse.issparse(X_valid_transformed):

    print("Output format:", X_valid_transformed.getformat())
    print("Output dtype:", X_valid_transformed.dtype)
    print("Non-zero values:", X_valid_transformed.nnz)

    total_values = (
        X_valid_transformed.shape[0] *
        X_valid_transformed.shape[1]
    )

    sparsity = 1 - (
        X_valid_transformed.nnz / total_values
    )

    print("Total matrix values:", total_values)
    print("Sparsity:", round(sparsity, 4))

    memory_mb = (
        X_valid_transformed.data.nbytes
        + X_valid_transformed.indices.nbytes
        + X_valid_transformed.indptr.nbytes
    ) / (1024 ** 2)

    print("Approximate sparse matrix memory:",
          round(memory_mb, 2), "MB")

else:
    print("WARNING: Output is dense.")

print("\n" + "=" * 60)
print("FULL VALIDATION TRANSFORMATION COMPLETED")
print("=" * 60)

TRANSFORMING FULL VALIDATION DATA

Input shape: (118108, 433)

Transformation completed.
Time taken: 0.10 minutes

Transformed shape: (118108, 768)
Output type: csr_matrix
Output format: csr
Output dtype: float64
Non-zero values: 51140764
Total matrix values: 90706944
Sparsity: 0.4362
Approximate sparse matrix memory: 585.71 MB

FULL VALIDATION TRANSFORMATION COMPLETED


In [15]:
# Step 28C — Generate Logistic Regression validation probabilities

import time
import numpy as np

print("=" * 60)
print("GENERATING LOGISTIC REGRESSION PREDICTIONS")
print("=" * 60)

print("\nValidation matrix shape:", X_valid_transformed.shape)
print("Validation target shape:", y_valid.shape)

start_time = time.time()

# Probability of fraud (class = 1)
y_valid_proba = logistic_model.predict_proba(
    X_valid_transformed
)[:, 1]

elapsed_time = time.time() - start_time

print("\nPrediction completed.")
print("Time taken: {:.2f} seconds".format(elapsed_time))

print("\nProbability array shape:", y_valid_proba.shape)
print("Minimum probability:", y_valid_proba.min())
print("Maximum probability:", y_valid_proba.max())
print("Mean probability:", y_valid_proba.mean())
print("Median probability:", np.median(y_valid_proba))

print("\nNaN probabilities:",
      np.isnan(y_valid_proba).sum())

print("Infinite probabilities:",
      np.isinf(y_valid_proba).sum())

print("\nFirst 10 fraud probabilities:")
print(np.round(y_valid_proba[:10], 6))

print("\n" + "=" * 60)
print("LOGISTIC REGRESSION PREDICTIONS COMPLETED")
print("=" * 60)

GENERATING LOGISTIC REGRESSION PREDICTIONS

Validation matrix shape: (118108, 768)
Validation target shape: (118108,)

Prediction completed.
Time taken: 0.27 seconds

Probability array shape: (118108,)
Minimum probability: 4.646752620632723e-32
Maximum probability: 1.0
Mean probability: 0.38380197546932626
Median probability: 0.3484206159775544

NaN probabilities: 0
Infinite probabilities: 0

First 10 fraud probabilities:
[0.65287  0.176395 0.953165 0.992318 0.242265 0.469071 0.296885 0.434652
 0.402252 0.151126]

LOGISTIC REGRESSION PREDICTIONS COMPLETED


In [16]:
# Step 28D — Evaluate Logistic Regression baseline

import time
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("=" * 60)
print("EVALUATING LOGISTIC REGRESSION BASELINE")
print("=" * 60)

# ------------------------------------------------------------
# Convert probabilities into class predictions
# using the standard 0.5 threshold
# ------------------------------------------------------------

threshold = 0.5

y_valid_pred = (y_valid_proba >= threshold).astype(int)

print("\nClassification threshold:", threshold)

# ------------------------------------------------------------
# Threshold-based metrics
# ------------------------------------------------------------

start_time = time.time()

precision = precision_score(
    y_valid,
    y_valid_pred,
    zero_division=0
)

recall = recall_score(
    y_valid,
    y_valid_pred,
    zero_division=0
)

f1 = f1_score(
    y_valid,
    y_valid_pred,
    zero_division=0
)

# ------------------------------------------------------------
# Threshold-independent metrics
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_valid,
    y_valid_proba
)

pr_auc = average_precision_score(
    y_valid,
    y_valid_proba
)

elapsed_time = time.time() - start_time

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

tn, fp, fn, tp = confusion_matrix(
    y_valid,
    y_valid_pred
).ravel()

print("\n" + "-" * 60)
print("THRESHOLD-BASED METRICS")
print("-" * 60)

print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

print("\n" + "-" * 60)
print("THRESHOLD-INDEPENDENT METRICS")
print("-" * 60)

print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\n" + "-" * 60)
print("CONFUSION MATRIX")
print("-" * 60)

print("True Negatives :", tn)
print("False Positives :", fp)
print("False Negatives :", fn)
print("True Positives  :", tp)

print("\n" + "-" * 60)
print("VALIDATION CLASSIFICATION REPORT")
print("-" * 60)

print(
    classification_report(
        y_valid,
        y_valid_pred,
        digits=4,
        zero_division=0
    )
)

print("Evaluation time: {:.2f} seconds".format(elapsed_time))

print("\n" + "=" * 60)
print("LOGISTIC REGRESSION BASELINE EVALUATION COMPLETED")
print("=" * 60)

EVALUATING LOGISTIC REGRESSION BASELINE

Classification threshold: 0.5

------------------------------------------------------------
THRESHOLD-BASED METRICS
------------------------------------------------------------
Precision: 0.0937
Recall: 0.7778
F1-score: 0.1673

------------------------------------------------------------
THRESHOLD-INDEPENDENT METRICS
------------------------------------------------------------
ROC-AUC: 0.8336
PR-AUC: 0.1808

------------------------------------------------------------
CONFUSION MATRIX
------------------------------------------------------------
True Negatives : 83470
False Positives : 30574
False Negatives : 903
True Positives  : 3161

------------------------------------------------------------
VALIDATION CLASSIFICATION REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

           0     0.9893    0.7319    0.8414    114044
           1     0.0937    0.7778    0.1673      40

In [17]:
# Step 28E — Logistic Regression threshold analysis

import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

print("=" * 60)
print("LOGISTIC REGRESSION THRESHOLD ANALYSIS")
print("=" * 60)

thresholds = np.arange(0.10, 0.91, 0.10)

threshold_results = []

for threshold in thresholds:

    y_pred_threshold = (
        y_valid_proba >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        y_pred_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_valid,
        y_pred_threshold,
        zero_division=0
    )

    f1 = f1_score(
        y_valid,
        y_pred_threshold,
        zero_division=0
    )

    flagged_count = y_pred_threshold.sum()

    flagged_percentage = (
        flagged_count / len(y_valid)
    ) * 100

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "flagged_transactions": flagged_count,
        "flagged_percentage": flagged_percentage
    })

threshold_results_df = pd.DataFrame(
    threshold_results
)

print("\nThreshold performance:")
print(
    threshold_results_df.to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "recall": "{:.4f}".format,
            "f1": "{:.4f}".format,
            "flagged_percentage": "{:.2f}%".format
        }
    )
)

print("\n" + "=" * 60)
print("THRESHOLD ANALYSIS COMPLETED")
print("=" * 60)

LOGISTIC REGRESSION THRESHOLD ANALYSIS

Threshold performance:
 threshold precision recall     f1  flagged_transactions flagged_percentage
       0.1    0.0372 0.9924 0.0717                108428             91.80%
       0.2    0.0460 0.9648 0.0878                 85213             72.15%
       0.3    0.0550 0.9186 0.1038                 67878             57.47%
       0.4    0.0705 0.8597 0.1304                 49538             41.94%
       0.5    0.0937 0.7778 0.1673                 33735             28.56%
       0.6    0.1281 0.6863 0.2159                 21769             18.43%
       0.7    0.1745 0.5768 0.2680                 13431             11.37%
       0.8    0.2270 0.4572 0.3033                  8186              6.93%
       0.9    0.2971 0.3423 0.3181                  4682              3.96%

THRESHOLD ANALYSIS COMPLETED


## Step 29 — Random Forest Baseline

Random Forest is introduced as a nonlinear tree-based benchmark.

Unlike Logistic Regression, Random Forest can capture nonlinear relationships
and interactions between features without requiring them to be explicitly
specified.

The model uses class weighting because fraudulent transactions represent only
a small proportion of the dataset.

The model will be evaluated on the same chronological validation set used for
Logistic Regression so that the comparison is fair.

Primary evaluation metrics:

- PR-AUC
- ROC-AUC
- Recall
- Precision
- F1-score

PR-AUC is particularly important because the fraud class is highly imbalanced.

In [18]:
# Step 29 — Random Forest benchmark

import time
import gc

from sklearn.ensemble import RandomForestClassifier

print("=" * 60)
print("TRAINING RANDOM FOREST")
print("=" * 60)

print("\nTraining matrix shape:", X_train_transformed.shape)
print("Training target shape:", y_train.shape)
print("Training fraud rate:", round(y_train.mean(), 6))

random_forest = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nModel configuration:")
print("Trees:", random_forest.n_estimators)
print("Maximum depth:", random_forest.max_depth)
print("Minimum samples per leaf:", random_forest.min_samples_leaf)
print("Class weight:", random_forest.class_weight)
print("Parallel jobs:", random_forest.n_jobs)

print("\nStarting training...")

start_time = time.time()

random_forest.fit(
    X_train_transformed,
    y_train
)

elapsed_time = time.time() - start_time

print("\nTraining completed.")
print("Time taken: {:.2f} minutes".format(elapsed_time / 60))

print("\n" + "=" * 60)
print("RANDOM FOREST TRAINING COMPLETED")
print("=" * 60)

TRAINING RANDOM FOREST

Training matrix shape: (472432, 768)
Training target shape: (472432,)
Training fraud rate: 0.035135

Model configuration:
Trees: 150
Maximum depth: 12
Minimum samples per leaf: 10
Class weight: balanced
Parallel jobs: -1

Starting training...

Training completed.
Time taken: 2.70 minutes

RANDOM FOREST TRAINING COMPLETED


In [19]:
# Step 29B — Generate Random Forest validation probabilities

import time
import numpy as np

print("=" * 60)
print("GENERATING RANDOM FOREST VALIDATION PREDICTIONS")
print("=" * 60)

print("\nValidation matrix shape:", X_valid_transformed.shape)
print("Validation target shape:", y_valid.shape)

start_time = time.time()

y_valid_proba_rf = random_forest.predict_proba(
    X_valid_transformed
)[:, 1]

elapsed_time = time.time() - start_time

print("\nPrediction completed.")
print("Time taken: {:.2f} seconds".format(elapsed_time))

print("\nProbability array shape:", y_valid_proba_rf.shape)

print("Minimum probability:", y_valid_proba_rf.min())
print("Maximum probability:", y_valid_proba_rf.max())
print("Mean probability:", y_valid_proba_rf.mean())
print("Median probability:", np.median(y_valid_proba_rf))

print("\nNaN probabilities:", np.isnan(y_valid_proba_rf).sum())
print("Infinite probabilities:", np.isinf(y_valid_proba_rf).sum())

print("\nFirst 10 fraud probabilities:")
print(np.round(y_valid_proba_rf[:10], 6))

print("\n" + "=" * 60)
print("RANDOM FOREST PREDICTIONS COMPLETED")
print("=" * 60)

GENERATING RANDOM FOREST VALIDATION PREDICTIONS

Validation matrix shape: (118108, 768)
Validation target shape: (118108,)

Prediction completed.
Time taken: 5.21 seconds

Probability array shape: (118108,)
Minimum probability: 0.05223005899331559
Maximum probability: 0.9969584363905823
Mean probability: 0.29029504643322845
Median probability: 0.2740294979889072

NaN probabilities: 0
Infinite probabilities: 0

First 10 fraud probabilities:
[0.434556 0.356058 0.584663 0.788339 0.220757 0.292741 0.221999 0.275348
 0.408461 0.126554]

RANDOM FOREST PREDICTIONS COMPLETED


In [20]:
# Step 29C — Evaluate Random Forest

import time
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("=" * 60)
print("EVALUATING RANDOM FOREST")
print("=" * 60)

threshold = 0.5

print("\nClassification threshold:", threshold)

start_time = time.time()

y_valid_pred_rf = (
    y_valid_proba_rf >= threshold
).astype(int)

precision_rf = precision_score(
    y_valid,
    y_valid_pred_rf,
    zero_division=0
)

recall_rf = recall_score(
    y_valid,
    y_valid_pred_rf,
    zero_division=0
)

f1_rf = f1_score(
    y_valid,
    y_valid_pred_rf,
    zero_division=0
)

roc_auc_rf = roc_auc_score(
    y_valid,
    y_valid_proba_rf
)

pr_auc_rf = average_precision_score(
    y_valid,
    y_valid_proba_rf
)

cm_rf = confusion_matrix(
    y_valid,
    y_valid_pred_rf
)

elapsed_time = time.time() - start_time

tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()

print("\n" + "-" * 50)
print("THRESHOLD-BASED METRICS")
print("-" * 50)

print("Precision:", round(precision_rf, 4))
print("Recall:", round(recall_rf, 4))
print("F1-score:", round(f1_rf, 4))

print("\n" + "-" * 50)
print("THRESHOLD-INDEPENDENT METRICS")
print("-" * 50)

print("ROC-AUC:", round(roc_auc_rf, 4))
print("PR-AUC:", round(pr_auc_rf, 4))

print("\n" + "-" * 50)
print("CONFUSION MATRIX")
print("-" * 50)

print("True Negatives :", tn_rf)
print("False Positives:", fp_rf)
print("False Negatives:", fn_rf)
print("True Positives  :", tp_rf)

flagged_rf = y_valid_pred_rf.sum()
flagged_percentage_rf = flagged_rf / len(y_valid) * 100

print("\nFlagged transactions:", flagged_rf)
print("Flagged percentage:", round(flagged_percentage_rf, 2), "%")

print("\n" + "-" * 50)
print("VALIDATION CLASSIFICATION REPORT")
print("-" * 50)

print(
    classification_report(
        y_valid,
        y_valid_pred_rf,
        digits=4,
        zero_division=0
    )
)

print("\nEvaluation time: {:.2f} seconds".format(elapsed_time))

print("\n" + "=" * 60)
print("RANDOM FOREST EVALUATION COMPLETED")
print("=" * 60)

EVALUATING RANDOM FOREST

Classification threshold: 0.5

--------------------------------------------------
THRESHOLD-BASED METRICS
--------------------------------------------------
Precision: 0.1921
Recall: 0.6567
F1-score: 0.2972

--------------------------------------------------
THRESHOLD-INDEPENDENT METRICS
--------------------------------------------------
ROC-AUC: 0.8709
PR-AUC: 0.4528

--------------------------------------------------
CONFUSION MATRIX
--------------------------------------------------
True Negatives : 102816
False Positives: 11228
False Negatives: 1395
True Positives  : 2669

Flagged transactions: 13897
Flagged percentage: 11.77 %

--------------------------------------------------
VALIDATION CLASSIFICATION REPORT
--------------------------------------------------
              precision    recall  f1-score   support

           0     0.9866    0.9015    0.9422    114044
           1     0.1921    0.6567    0.2972      4064

    accuracy                      

In [21]:
# Step 29D — Random Forest threshold analysis

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

print("=" * 60)
print("RANDOM FOREST THRESHOLD ANALYSIS")
print("=" * 60)

thresholds = np.arange(0.05, 1.00, 0.05)

threshold_results_rf = []

for threshold in thresholds:

    y_pred = (
        y_valid_proba_rf >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_valid,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_valid,
        y_pred,
        zero_division=0
    )

    flagged = y_pred.sum()
    flagged_percentage = flagged / len(y_valid) * 100

    false_negatives = (
        (y_valid == 1) & (y_pred == 0)
    ).sum()

    threshold_results_rf.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "flagged_transactions": flagged,
        "flagged_percentage": flagged_percentage,
        "false_negatives": false_negatives
    })

threshold_results_rf = pd.DataFrame(
    threshold_results_rf
)

print("\nThreshold performance:")

print(
    threshold_results_rf.to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "recall": "{:.4f}".format,
            "f1": "{:.4f}".format,
            "flagged_percentage": "{:.2f}%".format
        }
    )
)

print("\n" + "=" * 60)
print("HIGH-RECALL OPERATING POINTS")
print("=" * 60)

for recall_target in [0.90, 0.95, 0.99]:

    eligible = threshold_results_rf[
        threshold_results_rf["recall"] >= recall_target
    ]

    if len(eligible) > 0:

        best = eligible.sort_values(
            "threshold",
            ascending=False
        ).iloc[0]

        print(
            f"\nRecall >= {recall_target:.0%}:"
        )
        print(
            "Threshold:",
            best["threshold"]
        )
        print(
            "Recall:",
            round(best["recall"], 4)
        )
        print(
            "Precision:",
            round(best["precision"], 4)
        )
        print(
            "F1:",
            round(best["f1"], 4)
        )
        print(
            "Flagged transactions:",
            int(best["flagged_transactions"])
        )
        print(
            "Flagged percentage:",
            round(best["flagged_percentage"], 2),
            "%"
        )
        print(
            "False negatives:",
            int(best["false_negatives"])
        )

print("\n" + "=" * 60)
print("RANDOM FOREST THRESHOLD ANALYSIS COMPLETED")
print("=" * 60)

RANDOM FOREST THRESHOLD ANALYSIS

Threshold performance:
 threshold precision recall     f1  flagged_transactions flagged_percentage  false_negatives
      0.05    0.0344 1.0000 0.0665                118108            100.00%                0
      0.10    0.0381 0.9919 0.0734                105709             89.50%               33
      0.15    0.0460 0.9751 0.0879                 86118             72.91%              101
      0.20    0.0498 0.9523 0.0947                 77646             65.74%              194
      0.25    0.0560 0.9323 0.1057                 67650             57.28%              275
      0.30    0.0773 0.8787 0.1420                 46217             39.13%              493
      0.35    0.1004 0.8359 0.1793                 33834             28.65%              667
      0.40    0.1239 0.7874 0.2142                 25820             21.86%              864
      0.45    0.1557 0.7190 0.2560                 18764             15.89%             1142
      0.50   

In [22]:
# Step 30A — XGBoost boosting model

import time
import gc

from xgboost import XGBClassifier

print("=" * 60)
print("TRAINING XGBOOST")
print("=" * 60)

print("\nTraining matrix shape:", X_train_transformed.shape)
print("Training target shape:", y_train.shape)
print("Training fraud rate:", round(y_train.mean(), 6))

# Calculate the class imbalance ratio.
# This gives more importance to fraud observations during training.
scale_pos_weight = (
    (y_train == 0).sum() /
    (y_train == 1).sum()
)

print("\nClass imbalance configuration:")
print(
    "scale_pos_weight:",
    round(scale_pos_weight, 4)
)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

print("\nModel configuration:")
print("Trees:", xgb_model.n_estimators)
print("Maximum depth:", xgb_model.max_depth)
print("Learning rate:", xgb_model.learning_rate)
print("Row subsampling:", xgb_model.subsample)
print("Feature subsampling:", xgb_model.colsample_bytree)
print("Minimum child weight:", xgb_model.min_child_weight)
print("Evaluation metric:", xgb_model.eval_metric)
print("Tree method:", xgb_model.tree_method)
print("Parallel jobs:", xgb_model.n_jobs)

print("\nStarting training...")

start_time = time.time()

xgb_model.fit(
    X_train_transformed,
    y_train
)

elapsed_time = time.time() - start_time

print("\nTraining completed.")
print(
    "Time taken: {:.2f} minutes".format(
        elapsed_time / 60
    )
)

print("\n" + "=" * 60)
print("XGBOOST TRAINING COMPLETED")
print("=" * 60)

TRAINING XGBOOST

Training matrix shape: (472432, 768)
Training target shape: (472432,)
Training fraud rate: 0.035135

Class imbalance configuration:
scale_pos_weight: 27.4615

Model configuration:
Trees: 200
Maximum depth: 6
Learning rate: 0.08
Row subsampling: 0.8
Feature subsampling: 0.8
Minimum child weight: 5
Evaluation metric: aucpr
Tree method: hist
Parallel jobs: -1

Starting training...

Training completed.
Time taken: 0.92 minutes

XGBOOST TRAINING COMPLETED


In [23]:
# Step 30B — XGBoost validation predictions

import time
import numpy as np

print("=" * 60)
print("GENERATING XGBOOST VALIDATION PREDICTIONS")
print("=" * 60)

print("\nValidation matrix shape:", X_valid_transformed.shape)
print("Validation target shape:", y_valid.shape)

start_time = time.time()

y_valid_proba_xgb = xgb_model.predict_proba(
    X_valid_transformed
)[:, 1]

elapsed_time = time.time() - start_time

print(
    "\nPrediction completed. Time taken: {:.2f} seconds".format(
        elapsed_time
    )
)

print("\nProbability array shape:", y_valid_proba_xgb.shape)

print(
    "Minimum probability:",
    y_valid_proba_xgb.min()
)

print(
    "Maximum probability:",
    y_valid_proba_xgb.max()
)

print(
    "Mean probability:",
    y_valid_proba_xgb.mean()
)

print(
    "Median probability:",
    np.median(y_valid_proba_xgb)
)

print(
    "\nNaN probabilities:",
    np.isnan(y_valid_proba_xgb).sum()
)

print(
    "Infinite probabilities:",
    np.isinf(y_valid_proba_xgb).sum()
)

print(
    "\nFirst 10 fraud probabilities:"
)

print(
    np.round(
        y_valid_proba_xgb[:10],
        6
    )
)

print("\n" + "=" * 60)
print("XGBOOST PREDICTIONS COMPLETED")
print("=" * 60)

GENERATING XGBOOST VALIDATION PREDICTIONS

Validation matrix shape: (118108, 768)
Validation target shape: (118108,)

Prediction completed. Time taken: 2.34 seconds

Probability array shape: (118108,)
Minimum probability: 0.0016532887
Maximum probability: 0.9994512
Mean probability: 0.23278491
Median probability: 0.15996265

NaN probabilities: 0
Infinite probabilities: 0

First 10 fraud probabilities:
[0.307689 0.093651 0.866763 0.953126 0.104399 0.151641 0.281167 0.217572
 0.370913 0.115157]

XGBOOST PREDICTIONS COMPLETED


In [24]:
# Step 30C — XGBoost baseline evaluation

import time

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("=" * 60)
print("EVALUATING XGBOOST")
print("=" * 60)

threshold = 0.5

print("\nClassification threshold:", threshold)

start_time = time.time()

y_valid_pred_xgb = (
    y_valid_proba_xgb >= threshold
).astype(int)

precision = precision_score(
    y_valid,
    y_valid_pred_xgb,
    zero_division=0
)

recall = recall_score(
    y_valid,
    y_valid_pred_xgb,
    zero_division=0
)

f1 = f1_score(
    y_valid,
    y_valid_pred_xgb,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_valid,
    y_valid_proba_xgb
)

pr_auc = average_precision_score(
    y_valid,
    y_valid_proba_xgb
)

tn, fp, fn, tp = confusion_matrix(
    y_valid,
    y_valid_pred_xgb
).ravel()

flagged = y_valid_pred_xgb.sum()
flagged_percentage = flagged / len(y_valid) * 100

elapsed_time = time.time() - start_time

print("\n" + "-" * 50)
print("THRESHOLD-BASED METRICS")
print("-" * 50)

print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

print("\n" + "-" * 50)
print("THRESHOLD-INDEPENDENT METRICS")
print("-" * 50)

print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\n" + "-" * 50)
print("CONFUSION MATRIX")
print("-" * 50)

print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives  :", tp)

print("\nFlagged transactions:", flagged)
print(
    "Flagged percentage:",
    round(flagged_percentage, 2),
    "%"
)

print("\n" + "-" * 50)
print("VALIDATION CLASSIFICATION REPORT")
print("-" * 50)

print(
    classification_report(
        y_valid,
        y_valid_pred_xgb,
        digits=4,
        zero_division=0
    )
)

print("Evaluation time:", round(elapsed_time, 2), "seconds")

print("\n" + "=" * 60)
print("XGBOOST EVALUATION COMPLETED")
print("=" * 60)

EVALUATING XGBOOST

Classification threshold: 0.5

--------------------------------------------------
THRESHOLD-BASED METRICS
--------------------------------------------------
Precision: 0.2149
Recall: 0.7453
F1-score: 0.3336

--------------------------------------------------
THRESHOLD-INDEPENDENT METRICS
--------------------------------------------------
ROC-AUC: 0.9041
PR-AUC: 0.5099

--------------------------------------------------
CONFUSION MATRIX
--------------------------------------------------
True Negatives : 102975
False Positives: 11069
False Negatives: 1035
True Positives  : 3029

Flagged transactions: 14098
Flagged percentage: 11.94 %

--------------------------------------------------
VALIDATION CLASSIFICATION REPORT
--------------------------------------------------
              precision    recall  f1-score   support

           0     0.9900    0.9029    0.9445    114044
           1     0.2149    0.7453    0.3336      4064

    accuracy                         0.8

In [25]:
# Step 30D — XGBoost threshold analysis

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

print("=" * 60)
print("XGBOOST THRESHOLD ANALYSIS")
print("=" * 60)

thresholds = np.arange(0.05, 1.00, 0.05)

threshold_results_xgb = []

for threshold in thresholds:

    y_pred = (
        y_valid_proba_xgb >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_valid,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_valid,
        y_pred,
        zero_division=0
    )

    flagged = y_pred.sum()

    flagged_percentage = (
        flagged / len(y_valid)
    ) * 100

    false_negatives = (
        (y_valid == 1) &
        (y_pred == 0)
    ).sum()

    threshold_results_xgb.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "flagged_transactions": flagged,
        "flagged_percentage": flagged_percentage,
        "false_negatives": false_negatives
    })


threshold_results_xgb = pd.DataFrame(
    threshold_results_xgb
)


print("\nThreshold performance:")

print(
    threshold_results_xgb.to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "recall": "{:.4f}".format,
            "f1": "{:.4f}".format,
            "flagged_percentage": "{:.2f}%".format
        }
    )
)


print("\n" + "=" * 60)
print("HIGH-RECALL OPERATING POINTS")
print("=" * 60)


for recall_target in [0.90, 0.95, 0.99]:

    eligible = threshold_results_xgb[
        threshold_results_xgb["recall"] >= recall_target
    ]

    if len(eligible) > 0:

        best = eligible.sort_values(
            "threshold",
            ascending=False
        ).iloc[0]

        print(
            f"\nRecall >= {recall_target:.0%}:"
        )

        print(
            "Threshold:",
            best["threshold"]
        )

        print(
            "Recall:",
            round(best["recall"], 4)
        )

        print(
            "Precision:",
            round(best["precision"], 4)
        )

        print(
            "F1:",
            round(best["f1"], 4)
        )

        print(
            "Flagged transactions:",
            int(best["flagged_transactions"])
        )

        print(
            "Flagged percentage:",
            round(
                best["flagged_percentage"],
                2
            ),
            "%"
        )

        print(
            "False negatives:",
            int(best["false_negatives"])
        )


print("\n" + "=" * 60)
print("XGBOOST THRESHOLD ANALYSIS COMPLETED")
print("=" * 60)

XGBOOST THRESHOLD ANALYSIS

Threshold performance:
 threshold precision recall     f1  flagged_transactions flagged_percentage  false_negatives
      0.05    0.0384 0.9958 0.0739                105390             89.23%               17
      0.10    0.0486 0.9774 0.0926                 81723             69.19%               92
      0.15    0.0618 0.9515 0.1161                 62561             52.97%              197
      0.20    0.0791 0.9291 0.1457                 47763             40.44%              288
      0.25    0.0985 0.8967 0.1775                 36985             31.31%              420
      0.30    0.1190 0.8676 0.2093                 29622             25.08%              538
      0.35    0.1401 0.8383 0.2400                 24326             20.60%              657
      0.40    0.1624 0.8071 0.2704                 20197             17.10%              784
      0.45    0.1872 0.7751 0.3016                 16823             14.24%              914
      0.50    0.214

## Step 31 — LightGBM Challenger Model

LightGBM is evaluated as an independent gradient-boosting challenger
against XGBoost.

The objective is to determine whether LightGBM provides a meaningful
improvement in fraud-risk ranking on the same temporally separated
validation set.

The model is evaluated using PR-AUC as the primary metric because the
fraud class is highly imbalanced. ROC-AUC, precision, recall, and F1 are
also reported for comparison.

No threshold is selected at this stage. Threshold selection will be
performed only after the final model has been selected.

In [27]:
# Step 31A — Train LightGBM challenger

import time
import gc
import numpy as np

from lightgbm import LGBMClassifier

print("=" * 60)
print("TRAINING LIGHTGBM CHALLENGER")
print("=" * 60)

# Calculate class imbalance weight from the training data
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("\nTraining matrix shape:", X_train_transformed.shape)
print("Training fraud rate:", round(y_train.mean(), 6))
print("Scale positive weight:", round(scale_pos_weight, 4))

lightgbm_model = LGBMClassifier(
    objective="binary",

    n_estimators=200,
    learning_rate=0.08,
    max_depth=6,
    num_leaves=31,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_samples=20,

    scale_pos_weight=scale_pos_weight,

    random_state=42,
    n_jobs=-1,

    verbosity=-1
)

print("\nModel configuration:")
print("Estimators:", 200)
print("Learning rate:", 0.08)
print("Max depth:", 6)
print("Num leaves:", 31)
print("Scale positive weight:", round(scale_pos_weight, 4))

print("\nStarting training...")

start_time = time.time()

lightgbm_model.fit(
    X_train_transformed,
    y_train
)

elapsed_time = time.time() - start_time

print("\n" + "=" * 60)
print("LIGHTGBM TRAINING COMPLETED")
print("=" * 60)

print("Time taken:", round(elapsed_time / 60, 2), "minutes")
print("Number of trees:", lightgbm_model.n_estimators_)

TRAINING LIGHTGBM CHALLENGER

Training matrix shape: (472432, 768)
Training fraud rate: 0.035135
Scale positive weight: 27.4615

Model configuration:
Estimators: 200
Learning rate: 0.08
Max depth: 6
Num leaves: 31
Scale positive weight: 27.4615

Starting training...

LIGHTGBM TRAINING COMPLETED
Time taken: 0.54 minutes
Number of trees: 200


## LightGBM — Validation Predictions

The trained LightGBM model is applied to the temporally separated validation
set to generate fraud-risk scores.

The validation data is transformed using the preprocessing pipeline fitted
only on the training period.

At this stage, the predicted probabilities are treated primarily as
risk-ranking scores. No decision threshold is selected yet.

In [28]:
# Step 31B — Generate LightGBM validation predictions

import time
import numpy as np

print("=" * 60)
print("GENERATING LIGHTGBM VALIDATION PREDICTIONS")
print("=" * 60)

print("\nValidation matrix shape:", X_valid_transformed.shape)

start_time = time.time()

lightgbm_valid_proba = lightgbm_model.predict_proba(
    X_valid_transformed
)[:, 1]

elapsed_time = time.time() - start_time

print("\nPrediction completed.")
print("Time taken:", round(elapsed_time, 2), "seconds")

print("\nPrediction shape:", lightgbm_valid_proba.shape)
print("Minimum score:", round(lightgbm_valid_proba.min(), 6))
print("Maximum score:", round(lightgbm_valid_proba.max(), 6))
print("Mean score:", round(lightgbm_valid_proba.mean(), 6))
print("Median score:", round(np.median(lightgbm_valid_proba), 6))

print(
    "NaN values:",
    np.isnan(lightgbm_valid_proba).sum()
)

print(
    "Infinite values:",
    np.isinf(lightgbm_valid_proba).sum()
)

print("\nFirst 10 validation scores:")
print(lightgbm_valid_proba[:10])

print("\n" + "=" * 60)
print("LIGHTGBM VALIDATION PREDICTIONS COMPLETED")
print("=" * 60)

GENERATING LIGHTGBM VALIDATION PREDICTIONS

Validation matrix shape: (118108, 768)

Prediction completed.
Time taken: 1.86 seconds

Prediction shape: (118108,)
Minimum score: 0.0017
Maximum score: 0.999477
Mean score: 0.245472
Median score: 0.170032
NaN values: 0
Infinite values: 0

First 10 validation scores:
[0.36877625 0.06824314 0.89006873 0.94826511 0.05634395 0.06260773
 0.16905443 0.2302283  0.30770878 0.12062257]

LIGHTGBM VALIDATION PREDICTIONS COMPLETED


## LightGBM — Validation Evaluation

LightGBM is evaluated on the temporally separated validation period.

The evaluation includes:

- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC
- Confusion matrix
- Number and percentage of transactions flagged as fraud

A threshold of 0.5 is used only as a common reference point for comparing
models. The final operational threshold will be determined later using
business costs and investigation capacity.

In [29]:
# Step 31C — Evaluate LightGBM on validation data

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("=" * 60)
print("EVALUATING LIGHTGBM")
print("=" * 60)

# Common reference threshold only — not the final business threshold
reference_threshold = 0.5

lightgbm_valid_pred = (
    lightgbm_valid_proba >= reference_threshold
).astype(int)

# Classification metrics
precision = precision_score(
    y_valid,
    lightgbm_valid_pred,
    zero_division=0
)

recall = recall_score(
    y_valid,
    lightgbm_valid_pred,
    zero_division=0
)

f1 = f1_score(
    y_valid,
    lightgbm_valid_pred,
    zero_division=0
)

# Ranking metrics
roc_auc = roc_auc_score(
    y_valid,
    lightgbm_valid_proba
)

pr_auc = average_precision_score(
    y_valid,
    lightgbm_valid_proba
)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_valid,
    lightgbm_valid_pred
).ravel()

flagged_count = lightgbm_valid_pred.sum()
flagged_percentage = flagged_count / len(y_valid) * 100

print("\nReference threshold:", reference_threshold)

print("\nClassification metrics:")
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

print("\nRanking metrics:")
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nConfusion matrix:")
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

print("\nOperational view:")
print("Transactions flagged:", flagged_count)
print(
    "Percentage flagged:",
    round(flagged_percentage, 2),
    "%"
)

print("\nClassification report:")
print(
    classification_report(
        y_valid,
        lightgbm_valid_pred,
        digits=4,
        zero_division=0
    )
)

print("\n" + "=" * 60)
print("LIGHTGBM VALIDATION EVALUATION COMPLETED")
print("=" * 60)

EVALUATING LIGHTGBM

Reference threshold: 0.5

Classification metrics:
Precision: 0.1949
Recall: 0.7633
F1-score: 0.3105

Ranking metrics:
ROC-AUC: 0.9063
PR-AUC: 0.5106

Confusion matrix:
True Negatives : 101229
False Positives: 12815
False Negatives: 962
True Positives : 3102

Operational view:
Transactions flagged: 15917
Percentage flagged: 13.48 %

Classification report:
              precision    recall  f1-score   support

           0     0.9906    0.8876    0.9363    114044
           1     0.1949    0.7633    0.3105      4064

    accuracy                         0.8834    118108
   macro avg     0.5927    0.8255    0.6234    118108
weighted avg     0.9632    0.8834    0.9148    118108


LIGHTGBM VALIDATION EVALUATION COMPLETED


## Model Comparison

Four models were evaluated using the same temporally separated validation
period and the same feature representation:

1. Logistic Regression
2. Random Forest
3. XGBoost
4. LightGBM

PR-AUC is used as the primary ranking metric because the fraud class is
highly imbalanced. ROC-AUC, Precision, Recall, and F1 are also reported.

The 0.5 threshold is included only as a common reference point and is not
treated as the final operational decision threshold.

LightGBM achieved the highest PR-AUC and ROC-AUC, although its improvement
over XGBoost was marginal. XGBoost produced higher precision and F1 at the
reference threshold.

The final operational model decision will also consider ranking performance,
investigation capacity, threshold behavior, and business cost.

In [33]:
# Step 32 — Create final model comparison table

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],
    "PR-AUC": [
        0.1808,
        0.4528,
        0.5099,
        0.5106
    ],
    "ROC-AUC": [
        0.8336,
        0.8709,
        0.9041,
        0.9063
    ],
    "Precision @ 0.5": [
        0.0937,
        0.1921,
        0.2149,
        0.1949
    ],
    "Recall @ 0.5": [
        0.7778,
        0.6567,
        0.7453,
        0.7633
    ],
    "F1 @ 0.5": [
        0.1673,
        0.2972,
        0.3336,
        0.3105
    ]
})

print("=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

print(
    model_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n" + "=" * 80)
print("BEST MODELS BY METRIC")
print("=" * 80)

print(
    "Best PR-AUC:",
    model_comparison.loc[
        model_comparison["PR-AUC"].idxmax(),
        "Model"
    ]
)

print(
    "Best ROC-AUC:",
    model_comparison.loc[
        model_comparison["ROC-AUC"].idxmax(),
        "Model"
    ]
)

print(
    "Best Precision @ 0.5:",
    model_comparison.loc[
        model_comparison["Precision @ 0.5"].idxmax(),
        "Model"
    ]
)

print(
    "Best F1 @ 0.5:",
    model_comparison.loc[
        model_comparison["F1 @ 0.5"].idxmax(),
        "Model"
    ]
)

FINAL MODEL COMPARISON
              Model  PR-AUC  ROC-AUC  Precision @ 0.5  Recall @ 0.5  F1 @ 0.5
Logistic Regression  0.1808   0.8336           0.0937        0.7778    0.1673
      Random Forest  0.4528   0.8709           0.1921        0.6567    0.2972
            XGBoost  0.5099   0.9041           0.2149        0.7453    0.3336
           LightGBM  0.5106   0.9063           0.1949        0.7633    0.3105

BEST MODELS BY METRIC
Best PR-AUC: LightGBM
Best ROC-AUC: LightGBM
Best Precision @ 0.5: XGBoost
Best F1 @ 0.5: XGBoost


In [35]:
# Find probability/risk-score variables currently available

[x for x in globals().keys()
 if any(term in x.lower() for term in ["xgb", "xgboost", "proba", "probability"])]

['y_valid_proba',
 'y_valid_proba_rf',
 'XGBClassifier',
 'xgb_model',
 'y_valid_proba_xgb',
 'y_valid_pred_xgb',
 'threshold_results_xgb',
 'lightgbm_valid_proba']

In [36]:
# ============================================================
# STEP 33 — RANKING PERFORMANCE UNDER INVESTIGATION CAPACITY
# ============================================================

import pandas as pd
import numpy as np


def ranking_metrics(y_true, risk_scores, percentages):
    """
    Calculate ranking performance when investigators
    can review only a fixed percentage of transactions.
    """

    y_true = np.asarray(y_true)
    risk_scores = np.asarray(risk_scores)

    # Rank transactions from highest to lowest risk
    ranking_order = np.argsort(-risk_scores)

    total_transactions = len(y_true)
    total_fraud = y_true.sum()

    results = []

    for percentage in percentages:

        # Number of transactions that can be investigated
        k = max(1, int(total_transactions * percentage))

        # Select the highest-risk transactions
        top_indices = ranking_order[:k]

        # Count actual fraud cases in the investigation queue
        fraud_in_queue = y_true[top_indices].sum()

        precision_at_k = fraud_in_queue / k
        recall_at_k = fraud_in_queue / total_fraud

        results.append({
            "Investigation Capacity": f"Top {percentage * 100:.0f}%",
            "Transactions Investigated": k,
            "Fraud Found": int(fraud_in_queue),
            "Precision@K": precision_at_k,
            "Recall@K": recall_at_k,
            "Investigation Share": k / total_transactions
        })

    return pd.DataFrame(results)


# Investigation capacity scenarios
investigation_percentages = [
    0.01,
    0.05,
    0.10,
    0.20
]


# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------

lightgbm_ranking = ranking_metrics(
    y_valid,
    lightgbm_valid_proba,
    investigation_percentages
)


# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

xgboost_ranking = ranking_metrics(
    y_valid,
    y_valid_proba_xgb,
    investigation_percentages
)


# ------------------------------------------------------------
# Display LightGBM results
# ------------------------------------------------------------

print("=" * 80)
print("LIGHTGBM — RANKING PERFORMANCE")
print("=" * 80)

print(
    lightgbm_ranking.to_string(
        index=False,
        formatters={
            "Precision@K": "{:.4f}".format,
            "Recall@K": "{:.4f}".format,
            "Investigation Share": "{:.2%}".format
        }
    )
)


# ------------------------------------------------------------
# Display XGBoost results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("XGBOOST — RANKING PERFORMANCE")
print("=" * 80)

print(
    xgboost_ranking.to_string(
        index=False,
        formatters={
            "Precision@K": "{:.4f}".format,
            "Recall@K": "{:.4f}".format,
            "Investigation Share": "{:.2%}".format
        }
    )
)

LIGHTGBM — RANKING PERFORMANCE
Investigation Capacity  Transactions Investigated  Fraud Found Precision@K Recall@K Investigation Share
                Top 1%                       1181         1028      0.8704   0.2530               1.00%
                Top 5%                       5905         2261      0.3829   0.5563               5.00%
               Top 10%                      11810         2842      0.2406   0.6993              10.00%
               Top 20%                      23621         3397      0.1438   0.8359              20.00%

XGBOOST — RANKING PERFORMANCE
Investigation Capacity  Transactions Investigated  Fraud Found Precision@K Recall@K Investigation Share
                Top 1%                       1181         1030      0.8721   0.2534               1.00%
                Top 5%                       5905         2274      0.3851   0.5595               5.00%
               Top 10%                      11810         2865      0.2426   0.7050              10.00%
  

In [37]:
# ============================================================
# STEP 34 — COST-SENSITIVE OPERATING THRESHOLD ANALYSIS
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score


# ------------------------------------------------------------
# Thresholds to evaluate
# ------------------------------------------------------------

thresholds = np.arange(0.05, 1.00, 0.05)


# ------------------------------------------------------------
# Relative cost scenarios
#
# FP = unnecessary investigation of a legitimate transaction
# FN = missed fraudulent transaction
#
# These are illustrative relative costs, NOT actual monetary
# costs from the dataset or company.
# ------------------------------------------------------------

cost_scenarios = {
    "Balanced (1:1)": {
        "FP_cost": 1,
        "FN_cost": 1
    },
    "Fraud-sensitive (1:5)": {
        "FP_cost": 1,
        "FN_cost": 5
    },
    "Highly fraud-sensitive (1:10)": {
        "FP_cost": 1,
        "FN_cost": 10
    }
}


# ------------------------------------------------------------
# Calculate threshold metrics
# ------------------------------------------------------------

threshold_results = []


for threshold in thresholds:

    # Convert risk scores into binary decisions
    y_pred = (y_valid_proba_xgb >= threshold).astype(int)

    # Confusion matrix components
    tn = ((y_valid == 0) & (y_pred == 0)).sum()
    fp = ((y_valid == 0) & (y_pred == 1)).sum()
    fn = ((y_valid == 1) & (y_pred == 0)).sum()
    tp = ((y_valid == 1) & (y_pred == 1)).sum()

    flagged = tp + fp

    precision = precision_score(
        y_valid,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_valid,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_valid,
        y_pred,
        zero_division=0
    )

    row = {
        "Threshold": threshold,
        "Flagged Transactions": flagged,
        "Flagged %": flagged / len(y_valid),
        "Fraud Found": tp,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "False Positives": fp,
        "False Negatives": fn
    }

    # --------------------------------------------------------
    # Calculate relative costs for each scenario
    # --------------------------------------------------------

    for scenario_name, costs in cost_scenarios.items():

        relative_cost = (
            fp * costs["FP_cost"]
            + fn * costs["FN_cost"]
        )

        row[f"Cost — {scenario_name}"] = relative_cost

    threshold_results.append(row)


threshold_results = pd.DataFrame(threshold_results)


# ------------------------------------------------------------
# Display threshold analysis
# ------------------------------------------------------------

print("=" * 110)
print("XGBOOST — COST-SENSITIVE OPERATING THRESHOLD ANALYSIS")
print("=" * 110)

display(
    threshold_results.style.format({
        "Threshold": "{:.2f}",
        "Flagged %": "{:.2%}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}"
    })
)


# ------------------------------------------------------------
# Identify minimum-cost threshold under each scenario
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("MINIMUM RELATIVE COST BY SCENARIO")
print("=" * 110)

for scenario_name in cost_scenarios:

    cost_column = f"Cost — {scenario_name}"

    best_row = threshold_results.loc[
        threshold_results[cost_column].idxmin()
    ]

    print(f"\n{scenario_name}")
    print("-" * 50)
    print(f"Best threshold:        {best_row['Threshold']:.2f}")
    print(f"Flagged transactions:  {int(best_row['Flagged Transactions']):,}")
    print(f"Flagged share:         {best_row['Flagged %']:.2%}")
    print(f"Fraud found:           {int(best_row['Fraud Found']):,}")
    print(f"Precision:             {best_row['Precision']:.4f}")
    print(f"Recall:                {best_row['Recall']:.4f}")
    print(f"F1:                    {best_row['F1']:.4f}")
    print(f"False positives:       {int(best_row['False Positives']):,}")
    print(f"False negatives:       {int(best_row['False Negatives']):,}")
    print(f"Relative cost:         {int(best_row[cost_column]):,}")


print("\n" + "=" * 110)
print("COST-SENSITIVE THRESHOLD ANALYSIS COMPLETED")
print("=" * 110)

XGBOOST — COST-SENSITIVE OPERATING THRESHOLD ANALYSIS


,Threshold,Flagged Transactions,Flagged %,Fraud Found,Precision,Recall,F1,False Positives,False Negatives,Cost — Balanced (1:1),Cost — Fraud-sensitive (1:5),Cost — Highly fraud-sensitive (1:10)
0,0.05,105390,89.23%,4047,0.0384,0.9958,0.0739,101343,17,101360,101428,101513
1,0.10,81723,69.19%,3972,0.0486,0.9774,0.0926,77751,92,77843,78211,78671
2,0.15,62561,52.97%,3867,0.0618,0.9515,0.1161,58694,197,58891,59679,60664
3,0.20,47763,40.44%,3776,0.0791,0.9291,0.1457,43987,288,44275,45427,46867
4,0.25,36985,31.31%,3644,0.0985,0.8967,0.1775,33341,420,33761,35441,37541
5,0.30,29622,25.08%,3526,0.1190,0.8676,0.2093,26096,538,26634,28786,31476
6,0.35,24326,20.60%,3407,0.1401,0.8383,0.2400,20919,657,21576,24204,27489
7,0.40,20197,17.10%,3280,0.1624,0.8071,0.2704,16917,784,17701,20837,24757
8,0.45,16823,14.24%,3150,0.1872,0.7751,0.3016,13673,914,14587,18243,22813
9,0.50,14098,11.94%,3029,0.2149,0.7453,0.3336,11069,1035,12104,16244,21419



MINIMUM RELATIVE COST BY SCENARIO

Balanced (1:1)
--------------------------------------------------
Best threshold:        0.95
Flagged transactions:  1,411
Flagged share:         1.19%
Fraud found:           1,176
Precision:             0.8335
Recall:                0.2894
F1:                    0.4296
False positives:       235
False negatives:       2,888
Relative cost:         3,123

Fraud-sensitive (1:5)
--------------------------------------------------
Best threshold:        0.75
Flagged transactions:  4,989
Flagged share:         4.22%
Fraud found:           2,124
Precision:             0.4257
Recall:                0.5226
F1:                    0.4692
False positives:       2,865
False negatives:       1,940
Relative cost:         12,565

Highly fraud-sensitive (1:10)
--------------------------------------------------
Best threshold:        0.60
Flagged transactions:  9,785
Flagged share:         8.28%
Fraud found:           2,717
Precision:             0.2777
Recall:       

## Step 35 — Champion Model Selection

Four candidate models were evaluated using the same leakage-controlled
temporal validation period:

- Logistic Regression
- Random Forest
- XGBoost
- LightGBM

Model selection considered both global discrimination metrics and operational
performance under investigation-capacity constraints.

LightGBM achieved the highest overall PR-AUC and ROC-AUC, but its advantage
over XGBoost was marginal. XGBoost performed slightly better at the Top 1%,
Top 5%, and Top 10% investigation capacities evaluated.

Therefore, XGBoost is selected as the champion model for the downstream risk
evaluation stage.

The validation period has been used for model comparison and operating-point
analysis. Consequently, it should not be treated as an untouched final test
set. A separate future holdout would be required for unbiased final
performance estimation before production deployment.

Threshold selection is treated as a business decision rather than a fixed
model property. The appropriate operating threshold depends on fraud-loss
severity, false-positive/customer-friction costs, investigation capacity,
and other operational constraints.

In [38]:
# ============================================================
# STEP 35A — FINAL CHAMPION MODEL RECORD
# ============================================================

champion_model_name = "XGBoost"

# Freeze the selected model for downstream evaluation
champion_model = xgb_model

# Freeze the corresponding validation risk scores
champion_valid_proba = y_valid_proba_xgb

champion_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],
    "PR-AUC": [
        0.1808,
        0.4528,
        0.5099,
        0.5106
    ],
    "ROC-AUC": [
        0.8336,
        0.8709,
        0.9041,
        0.9063
    ],
    "Decision": [
        "Baseline",
        "Challenger",
        "Champion",
        "Challenger"
    ]
})

print("=" * 90)
print("FINAL CHAMPION MODEL")
print("=" * 90)

display(champion_summary)

print("\nSelected model:", champion_model_name)
print("Model type:", type(champion_model).__name__)
print("Validation risk scores:", champion_valid_proba.shape)
print("Validation transactions:", len(y_valid))
print("Validation fraud rate:", f"{y_valid.mean():.4%}")

print("\nCHAMPION MODEL FINALIZED")

FINAL CHAMPION MODEL


,Model,PR-AUC,ROC-AUC,Decision
0,Logistic Regression,0.1808,0.8336,Baseline
1,Random Forest,0.4528,0.8709,Challenger
2,XGBoost,0.5099,0.9041,Champion
3,LightGBM,0.5106,0.9063,Challenger



Selected model: XGBoost
Model type: XGBClassifier
Validation risk scores: (118108,)
Validation transactions: 118108
Validation fraud rate: 3.4409%

CHAMPION MODEL FINALIZED


## Step 35B — Champion Model Risk Scores

The selected XGBoost champion model produces a continuous risk score for each
transaction in the validation period.

These scores, together with the transaction identifier, transaction time, and
observed fraud outcome, will be saved for downstream risk evaluation.

The risk evaluation notebook will use these outputs to analyse risk
concentration, investigation capacity, operating thresholds, and business
trade-offs.

In [40]:
# ============================================================
# STEP 35B — SAVE CHAMPION RISK SCORES
# ============================================================

import pandas as pd
import numpy as np
import time

# ------------------------------------------------------------
# Reload only traceability columns from raw transaction data
# ------------------------------------------------------------
# TransactionID is not a model feature.
# We reload it separately so the risk scores can be traced
# back to individual transactions.

start_time = time.time()

trace_df = pd.read_csv(
    "../data/raw/train_transaction.csv",
    usecols=["TransactionID", "TransactionDT"]
)

# Recreate the exact chronological ordering used for the
# train/validation split.

trace_df = (
    trace_df
    .sort_values("TransactionDT")
    .reset_index(drop=True)
)

# Extract the same final 20% validation period.

trace_valid = trace_df.iloc[
    len(trace_df) - len(X_valid):
].copy()

elapsed_time = time.time() - start_time

print("=" * 90)
print("VALIDATION TRACEABILITY DATA")
print("=" * 90)

print("Raw traceability shape:", trace_df.shape)
print("Validation traceability shape:", trace_valid.shape)
print("Time taken:", f"{elapsed_time:.2f} seconds")

# ------------------------------------------------------------
# Verify exact row alignment
# ------------------------------------------------------------

transaction_dt_match = np.array_equal(
    trace_valid["TransactionDT"].values,
    X_valid["TransactionDT"].values
)

print("\nTransactionDT alignment:", transaction_dt_match)

# ------------------------------------------------------------
# Create final champion risk-score dataset
# ------------------------------------------------------------

champion_risk_scores = pd.DataFrame({
    "TransactionID": trace_valid["TransactionID"].values,
    "TransactionDT": trace_valid["TransactionDT"].values,
    "isFraud": y_valid.values,
    "XGBoost_Risk_Score": champion_valid_proba
})

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CHAMPION RISK SCORE DATASET")
print("=" * 90)

print("Shape:", champion_risk_scores.shape)

print("\nColumns:")
print(champion_risk_scores.columns.tolist())

print("\nMissing values:")
print(champion_risk_scores.isna().sum())

print("\nRisk score range:")
print(
    "Minimum:",
    champion_risk_scores["XGBoost_Risk_Score"].min()
)

print(
    "Maximum:",
    champion_risk_scores["XGBoost_Risk_Score"].max()
)

print("\nTarget distribution:")
print(champion_risk_scores["isFraud"].value_counts())

print("\nRow alignment:")
print("Rows:", len(champion_risk_scores))
print("Risk scores:", len(champion_valid_proba))
print("Target:", len(y_valid))

# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

assert transaction_dt_match, "TransactionDT alignment failed."
assert len(champion_risk_scores) == len(champion_valid_proba)
assert len(champion_risk_scores) == len(y_valid)
assert champion_risk_scores["TransactionID"].is_unique
assert champion_risk_scores.isna().sum().sum() == 0

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output_path = "../data/processed/champion_risk_scores.csv"

champion_risk_scores.to_csv(
    output_path,
    index=False
)

print("\nSaved to:")
print(output_path)

print("\n" + "=" * 90)
print("CHAMPION RISK SCORE DATASET SAVED")
print("=" * 90)

VALIDATION TRACEABILITY DATA
Raw traceability shape: (590540, 2)
Validation traceability shape: (118108, 2)
Time taken: 4.18 seconds

TransactionDT alignment: True

CHAMPION RISK SCORE DATASET
Shape: (118108, 4)

Columns:
['TransactionID', 'TransactionDT', 'isFraud', 'XGBoost_Risk_Score']

Missing values:
TransactionID         0
TransactionDT         0
isFraud               0
XGBoost_Risk_Score    0
dtype: int64

Risk score range:
Minimum: 0.0016532887
Maximum: 0.9994512

Target distribution:
isFraud
0    114044
1      4064
Name: count, dtype: int64

Row alignment:
Rows: 118108
Risk scores: 118108
Target: 118108

Saved to:
../data/processed/champion_risk_scores.csv

CHAMPION RISK SCORE DATASET SAVED


In [41]:
# ============================================================
# STEP 41A — SAVE CHAMPION MODEL ARTIFACTS
# ============================================================

import os
import joblib
import time

start_time = time.time()

# ------------------------------------------------------------
# 1. Create model artifact directory
# ------------------------------------------------------------

artifact_dir = "../data/processed/model_artifacts"

os.makedirs(
    artifact_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Save fitted preprocessing pipeline
# ------------------------------------------------------------

preprocessor_path = (
    f"{artifact_dir}/xgboost_preprocessor.joblib"
)

joblib.dump(
    final_preprocessor,
    preprocessor_path
)

# ------------------------------------------------------------
# 3. Save trained champion XGBoost model
# ------------------------------------------------------------

model_path = (
    f"{artifact_dir}/xgboost_champion_model.joblib"
)

joblib.dump(
    champion_model,
    model_path
)

# ------------------------------------------------------------
# 4. Save transformed feature names
# ------------------------------------------------------------

feature_names_path = (
    f"{artifact_dir}/xgboost_feature_names.joblib"
)

feature_names = final_preprocessor.get_feature_names_out()

joblib.dump(
    feature_names,
    feature_names_path
)

elapsed_time = time.time() - start_time

# ------------------------------------------------------------
# 5. Verify artifacts
# ------------------------------------------------------------

print("=" * 100)
print("CHAMPION MODEL ARTIFACTS")
print("=" * 100)

print("Preprocessor saved:")
print(preprocessor_path)

print("\nChampion model saved:")
print(model_path)

print("\nFeature names saved:")
print(feature_names_path)

print("\nTransformed feature count:", len(feature_names))

print("\nArtifact files:")
for filename in sorted(os.listdir(artifact_dir)):
    print("-", filename)

print("\nTime taken:", f"{elapsed_time:.2f} seconds")

print("\nCHAMPION MODEL ARTIFACTS SAVED")

CHAMPION MODEL ARTIFACTS
Preprocessor saved:
../data/processed/model_artifacts/xgboost_preprocessor.joblib

Champion model saved:
../data/processed/model_artifacts/xgboost_champion_model.joblib

Feature names saved:
../data/processed/model_artifacts/xgboost_feature_names.joblib

Transformed feature count: 768

Artifact files:
- xgboost_champion_model.joblib
- xgboost_feature_names.joblib
- xgboost_preprocessor.joblib

Time taken: 0.06 seconds

CHAMPION MODEL ARTIFACTS SAVED
